# Reconstructed seventh-place preprocessing
Source: https://github.com/andreiolariu/kaggle-event-recommendation

Original multi-scale clustering, location reconstruction and history features. Uses 3,137,972 events; cluster sizes 8, 20 and 40. No training labels enter preprocessing.

In [ ]:
import ast
import csv
import gzip
import json
import pickle
from collections import Counter, defaultdict
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.cluster import MiniBatchKMeans
from threadpoolctl import threadpool_limits

SIZES = [8, 20, 40]  # Preserve the original three clustering resolutions.



### Adapt historical collection lookups


In [ ]:
class Collection:
    def __init__(self, records, key):
        self.records, self.key = records, key
    def find_one(self, query):
        return self.records.get(int(query[self.key]))
    def find(self, query=None):
        if query is None:
            return list(self.records.values())
        value = query[self.key]
        ids = value['$in'] if isinstance(value, dict) else [value]
        return [self.records[int(i)] for i in ids if int(i) in self.records]



### Index historical participation records


In [ ]:
class Attendance:
    def __init__(self, by_user, by_event, positives):
        self.by_user, self.by_event, self.positives = by_user, by_event, positives
    def find(self, query):
        if 'uid' in query:
            return self.by_user.get(int(query['uid']), [])
        eid = int(query['eid'])
        records = {}
        for key, actors in self.by_event.get(eid, {}).items():
            for actor in actors:
                records.setdefault(actor, {'uid': actor, 'eid': eid})[key] = True
        return list(records.values())
    def positive_users(self, eid):
        return self.positives.get(int(eid), set())



### Load or rebuild historical preprocessing


In [ ]:
def load_collections():
    OUT.mkdir(parents=True, exist_ok=True)
    cache = OUT / 'preprocessed.pkl'
    if cache.exists():
        with cache.open('rb') as stream:
            u, e, f, au, ae, positive = pickle.load(stream)
    else:
        with threadpool_limits(limits=4):
            u, e, f, au, ae, positive = build()
        with cache.open('xb') as stream:
            pickle.dump((u, e, f, au, ae, positive), stream, protocol=5)
    return Collection(u, 'id'), Collection(e, 'id'), Attendance(au, ae, positive), Collection(f, 'uid')



### Reconstruct clusters and historical user profiles


In [ ]:
def build():
    raw = ROOT / 'data/raw'
    train = pd.read_csv(raw / 'train.csv', usecols=['user', 'event'])
    targets = set(train.user.astype(int))
    candidate_ids = set(train.event.astype(int))
    users = pd.read_csv(raw / 'users.csv', dtype=str).fillna('')
    friends = {}
    with gzip.open(raw / 'user_friends.csv.gz', 'rt', newline='') as stream:
        for row in csv.DictReader(stream):
            uid = int(row['user'])
            if uid in targets:
                friends[uid] = {'uid': uid, 'friends': sorted(set(map(int, row['friends'].split())))}
    people = np.array(sorted(targets | {i for f in friends.values() for i in f['friends']}), dtype=np.int64)
    for uid in targets:
        friends.setdefault(uid, {'uid': uid, 'friends': []})
    people_index = pd.Index(people)
    counts = [f'c_{i}' for i in range(1, 101)] + ['c_other']
    n = 3137972
    words = np.lib.format.open_memmap(OUT / 'event_words.npy', mode='w+', dtype='float32', shape=(n, 101))
    ids, creators, codes = np.empty(n, np.int64), np.empty(n, np.int64), np.empty(n, np.int32)
    location_ids, location_list, grid = {}, [], defaultdict(set)
    coordinate = np.full((n, 2), np.nan, np.float32)
    starts = {}
    kms = [MiniBatchKMeans(n_clusters=k, batch_size=100, n_init=3, random_state=42) for k in SIZES]
    offset = 0
    for chunk in pd.read_csv(raw / 'events.csv.gz', chunksize=10000, dtype={c: np.float32 for c in counts}):
        stop = offset + len(chunk)  # End offset of the current batch.
        block = chunk[counts].to_numpy(dtype=np.float32)  # Keep the original raw counts for historical clustering.
        words[offset:stop] = block
        ids[offset:stop], creators[offset:stop] = chunk.event_id, chunk.user_id
        coordinate[offset:stop] = chunk[['lat', 'lng']].to_numpy(dtype=np.float32)
        labels = chunk[['city', 'country', 'state']].fillna('').astype(str)
        for j, loc in enumerate(labels.itertuples(index=False, name=None)):
            loc = tuple(None if not v else v for v in loc)
            loc = (loc[0], 'Democratic Republic of the Congo' if loc[1] == 'Democratic Republic Congo' else loc[1], loc[2])
            if loc not in location_ids:
                location_ids[loc] = len(location_list)
                location_list.append(loc)
            codes[offset + j] = location_ids[loc]
            lat, lng = coordinate[offset + j]
            if loc[0] and np.isfinite(lat) and np.isfinite(lng) and lat != 0:
                grid[(int(float(lat) * 5), int(float(lng) * 5))].add(location_ids[loc])  # Use integer geographic grid keys to avoid floating-point lookup errors.
        for row in chunk.loc[chunk.event_id.isin(candidate_ids), ['event_id', 'start_time']].itertuples(index=False):
            starts[int(row.event_id)] = pd.Timestamp(row.start_time).timestamp()
        if len(chunk) == 10000:  # Match cluster.py: do not fit the final batch if it has fewer than 10,000 rows.
            for km in kms:
                km.partial_fit(block)
        offset = stop
        if offset % 500000 == 0:
            print('活动聚类预处理:', offset, flush=True)
    words.flush()
    event_index = pd.Index(ids)
    clusters = np.empty((n, 3), np.int16)
    for start in range(0, n, 50000):
        for j, km in enumerate(kms):
            clusters[start:start + 50000, j] = km.predict(words[start:start + 50000])
    np.save(OUT / 'event_clusters.npy', clusters)
    print('聚类完成，开始关系预处理', flush=True)
    history_rows, history_cols = [[], [], []], [[], [], []]
    event_status = {}
    with (raw / 'event_attendees.csv').open(newline='') as stream:
        for row in csv.DictReader(stream):
            eid = int(row['event'])
            pos = event_index.get_indexer([eid])[0]
            status = {k: set(map(int, row[k].split())) for k in ['yes', 'no', 'maybe', 'invited']}
            event_status[eid] = status
            if pos < 0:
                continue
            for group, actors in enumerate([status['yes'] | status['maybe'], status['no'], status['invited']]):  # Positive history includes yes and maybe responses.
                index = people_index.get_indexer(list(actors))
                index = index[index >= 0]
                history_rows[group].extend(index.tolist())
                history_cols[group].extend([pos] * len(index))
    creator_rows = people_index.get_indexer(creators)  # The original setup.py also treats event creators as participants.
    valid = np.flatnonzero(creator_rows >= 0)
    history_rows[0].extend(creator_rows[valid].tolist())
    history_cols[0].extend(valid.tolist())
    histories = []
    for rows, cols in zip(history_rows, history_cols):
        matrix = sparse.csr_matrix((np.ones(len(rows), np.float32), (rows, cols)), shape=(len(people), n))
        matrix.data[:] = 1  # Count each user-event relationship once.
        histories.append(matrix)
    del history_rows, history_cols
    def events_for(group, actors):
        idx = people_index.get_indexer(list(actors))
        idx = idx[idx >= 0]
        return np.unique(histories[group][idx].indices)
    def taste(indices):
        return {f'cl{j}': (np.bincount(clusters[indices, j], minlength=k) / len(indices)).tolist() for j, k in enumerate(SIZES)}
    user_locations, event_locations = locations(users, codes, location_list, coordinate, grid, event_index, friends, histories[0], people_index, candidate_ids)
    user_docs, event_docs, by_user, by_event, positive = {}, {}, {}, {}, {}
    relevant_events = set(candidate_ids)
    for number, uid in enumerate(sorted(targets), 1):
        fids = friends.get(uid, {'friends': []})['friends']
        doc = {'id': uid, 'birth': 'None', 'gender': 'None', 'location': None, 'newloc2': user_locations.get(uid, [])}
        own = [events_for(j, [uid]) for j in range(3)]
        friend_pos, friend_neg = events_for(0, fids), events_for(1, fids)
        for key, indices in zip(['user_taste', 'friends_taste', 'user_hates', 'friends_hate', 'user_invited'], [own[0], friend_pos, own[1], friend_neg, own[2]]):
            if len(indices):
                doc[key] = taste(indices)
        proto_ids = np.union1d(own[0], friend_pos)  # The historical content prototype includes both the user and friends.
        if len(proto_ids):
            prototype = words[proto_ids].sum(axis=0, dtype=np.float64)
            if prototype.sum() > 0:
                doc['prototype'] = (prototype / prototype.sum()).tolist()
        user_docs[uid] = doc
        by_user[uid] = [{'uid': uid, 'eid': int(ids[i]), 'yes': True} for i in own[0]]
        relevant_events.update(int(ids[i]) for i in own[0])
        if number % 250 == 0:
            print('用户偏好:', number, '/', len(targets), flush=True)
    for eid in relevant_events:
        index = event_index.get_indexer([eid])[0]
        status = event_status.get(eid, {'yes': set(), 'no': set(), 'maybe': set(), 'invited': set()})
        positives = status['yes'].copy()  # Event similarity uses yes responses, excluding maybe.
        if index >= 0:
            positives.add(int(creators[index]))
        positive[eid] = positives
        if eid in candidate_ids:
            by_event[eid] = dict(status, yes=positives)
            event_docs[eid] = {'id': eid, 'creator': int(creators[index]), 'words': words[index].tolist(), 'start': starts[eid], 'location': None, 'newloc2': event_locations.get(eid, []), **{f'cl{j}': int(clusters[index, j]) for j in range(3)}}
    audit = {'source_rows': n, 'target_users': len(targets), 'candidate_events': len(candidate_ids), 'relevant_people': len(people), 'location_users_covered': sum(bool(user_docs[u]['newloc2']) for u in user_docs), 'location_events_covered': sum(bool(e['newloc2']) for e in event_docs.values()), 'missing_profiles': {k: sum(k not in u for u in user_docs.values()) for k in ['friends_taste', 'friends_hate', 'user_invited', 'prototype']}, 'label_policy': '预处理从不读取 interested/not_interested；不注入训练或公开测试答案。', 'cluster_sizes': SIZES, 'seed': 42}
    (OUT / 'preprocess_audit.json').write_text(json.dumps(audit, ensure_ascii=False, indent=2))
    return user_docs, event_docs, friends, by_user, by_event, positive



### Resolve locations from competition metadata


In [ ]:
def locations(users, codes, locs, coordinates, grid, event_index, friends, history, people_index, candidates):
    constants = json.loads((HERE / 'regions.json').read_text())  # Read the original geographic lookup constants.
    countries = {v[1] for v in locs if v[1]}
    cities = defaultdict(set)
    for city, country, state in locs:
        if city and country:
            cities[city].add(country)
    rev = {name: code for key in ['us_states', 'canada_states'] for code, name in constants[key].items()}
    def as_doc(loc):
        return {k: v for k, v in zip(['city', 'country', 'state'], [loc[0], loc[1], rev.get(loc[2], loc[2])]) if v}
    def event_loc(index):
        lat, lng = coordinates[index]
        matches = None
        if np.isfinite(lat) and np.isfinite(lng) and lat != 0 and lng != 0:
            cell = (int(float(lat) * 5), int(float(lng) * 5))
            matches = grid.get(cell)
            if not matches:
                best = 5.0
                for a in range(-10, 11, 2):
                    for b in range(-10, 11, 2):
                        nearby = (cell[0] + a, cell[1] + b)
                        distance = np.hypot(nearby[0] / 5 - lat, nearby[1] / 5 - lng)
                        if nearby in grid and distance < best:
                            matches, best = grid[nearby], distance  # Use the current neighboring cell instead of the stale variable in the original snippet.
        values = [locs[i] for i in sorted(matches)] if matches else [locs[codes[index]]]
        return [as_doc(v) for v in values if any(v)]
    resolved = {}
    for row in users.itertuples(index=False):
        tokens = [t.strip() for t in row.location.split('  ') if t.strip() and t.strip() != 'undefined' and not t.strip().isdigit()]
        country, state, city = next((t for t in tokens if t in countries), None), None, None
        for token in tokens:
            for key, nation in [('us_states', 'United States'), ('canada_states', 'Canada')]:
                if token in constants[key] or token in constants[key].values():
                    country, state = nation, rev.get(token, token)
            if token in constants['australia_states']:
                country, state = 'Australia', token
        if country:
            city = next((t for t in reversed(tokens) if country in cities.get(t, set())), None)
        if not (country or state or city):
            remaining = set(countries)
            for token in tokens:
                options = set(cities.get(token, set()))
                for nation in ['United States', 'Canada', 'Australia']:
                    if len(options) > 1:
                        options.discard(nation)
                if len(options) == 1:
                    city, country = token, next(iter(options))
                elif options:
                    remaining &= options
            if country is None and len(remaining) < 5:
                country = sorted(remaining)
        country = country or tokens
        nations = [country] if isinstance(country, str) else country
        resolved[int(row.user_id)] = [as_doc((city, nation, state)) for nation in nations]
    initial = dict(resolved)  # Freeze the initial locations before friend voting.
    event_cache = {}
    for uid in sorted(friends):
        if resolved.get(uid):
            continue
        index = people_index.get_indexer([uid])[0]
        past = history[index].indices if index >= 0 else []
        votes = Counter()
        for i in past:
            if int(i) not in event_cache:
                event_cache[int(i)] = event_loc(int(i))
            for loc in event_cache[int(i)]:
                votes[tuple(loc.get(k) for k in ['city', 'country', 'state'])] += 1
        for fid in friends[uid]['friends']:
            for loc in initial.get(fid, []):  # Use the original locations so inference does not depend on processing order.
                votes[tuple(loc.get(k) for k in ['city', 'country', 'state'])] += 1
        ranked = votes.most_common()
        stop = 1
        while len(ranked) > stop and ranked[stop - 1][1] < ranked[stop][1] * 4:  # Stop when adjacent location vote counts differ by a factor of four.
            stop += 1
        resolved[uid] = [as_doc(loc) for loc, count in ranked[:stop]]
    event_resolved = {}
    for eid in candidates:
        index = event_index.get_indexer([eid])[0]
        event_resolved[eid] = event_cache.get(index) or event_loc(index)
    return resolved, event_resolved
